# Allocentric social place fields — demo

Compute occupancy-normalized rate maps of a focal animal's cells over each
(self and partner) animal's allocentric `(x, y)`, then look at a few example
cells and the population classification.

See [`ephys/social_spatial_fields.py`](social_spatial_fields.py) for the API and
the project `README.md` / `CLAUDE.md` for the gotchas (occupancy is over the
**target**; tracking↔ephys conversion lives only in
`MultiAnimalSession.get_tracking_on_ephys_clock`).

In [ ]:
import sys, os
sys.path.append(os.path.abspath('..'))
%matplotlib inline

from ingestion.multi_animal_session import MultiAnimalSession
from ephys.social_spatial_fields import compute_social_place_fields
from ephys import social_spatial_plots as sp

## 1. Load a session and compute social place fields

Edit `SESSION`, `ANIMALS`, and `FOCAL` for your data. The focal animal's spikes
generate the maps; every animal in `ANIMALS` (including the focal) is a target.

In [ ]:
SESSION = '20251216'
ANIMALS = ['631', '632', '633']
FOCAL = '631'

session = MultiAnimalSession(session_id=SESSION, animal_ids=ANIMALS)
ks = session.get_ks(FOCAL)

results = compute_social_place_fields(
    ks, session, focal_animal=FOCAL, target_animals=ANIMALS,
    bin_size_cm=5.0, smoothing_sigma_cm=5.0,
    speed_threshold_cms=5.0, speed_filter_subject='target',
    n_shuffles=200, use_quality_cells=True,
)
results.cell_classification['category'].value_counts()

## 2. Rate-map grids for a few example cells

Top cells by max Skaggs bits/spike across targets. One panel per target animal
(the focal target is the self place field).

In [ ]:
import numpy as np
df = results.cell_classification
bits_cols = [f'bits_per_spike_{t}' for t in ANIMALS if f'bits_per_spike_{t}' in df.columns]
top = df.assign(_m=df[bits_cols].max(axis=1)).sort_values('_m', ascending=False)
top_clusters = top['cluster_id'].head(3).astype(int).tolist()

for cid in top_clusters:
    sp.plot_rate_maps_grid(results, cluster_id=cid)

## 3. Population classification summary

In [ ]:
sp.plot_cell_classification_summary(results)
sp.plot_field_stability(results)